# 💻 Chapter 13: Probability in Code — Simulating Randomness
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Your `random()` function isn't truly random — and that's actually great.
> Understanding how it works makes you a better engineer.

**🎯 Objectives:** Understand PRNGs, seeds, reproducibility, and entropy sources.

## 📖 Section 1 — Concept Review

### PRNG vs TRNG
| | **PRNG** (Pseudo-Random) | **TRNG** (True Random) |
|--|--|--|
| Source | Algorithm (deterministic) | Physical entropy (hardware) |
| Speed | Very fast | Slow |
| Reproducible | Yes (with seed) | No |
| Quality | Good algorithms are excellent | Perfect |
| Use | Simulations, ML, games | Cryptography, key generation |

### The Mersenne Twister (numpy's default)
- Period: 2^19937 - 1 (astronomically large)
- Passes most statistical tests
- **NOT cryptographically secure** — don't use for passwords!

### Seeds and Reproducibility
```python
np.random.seed(42)  # Global seed — reproducible
rng = np.random.default_rng(42)  # Better: Generator object (recommended)
```

### NumPy's Newer API (Recommended)
```python
rng = np.random.default_rng(seed=42)  # PCG64 algorithm — better quality
x = rng.uniform(0, 1, 1000)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import time
sns.set_theme(style="whitegrid")

# ── PRNG vs Seed ──
print("🔢 Understanding Seeds and Reproducibility")
print()

# Without seed: different every time
print("Without seed (first 5):", np.random.randint(0, 100, 5))
print("Without seed (next 5): ", np.random.randint(0, 100, 5))
print()

# With seed: reproducible
np.random.seed(42)
print("With seed=42 (run 1):", np.random.randint(0, 100, 5))
np.random.seed(42)
print("With seed=42 (run 2):", np.random.randint(0, 100, 5))  # same!
print()

# Modern API (recommended)
rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(42)
print("PCG64 rng1:", rng1.integers(0, 100, 5))
print("PCG64 rng2:", rng2.integers(0, 100, 5))  # same!
print()
print("💡 Use rng = np.random.default_rng(seed) in new code!")
print("   It uses the newer, better PCG64 algorithm.")

In [ ]:
# ── Test PRNG Quality: Statistical Tests ──
rng = np.random.default_rng(42)
n = 100_000
samples = rng.uniform(0, 1, n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Uniformity test
axes[0].hist(samples, bins=50, color='#3498db', edgecolor='white', density=True)
axes[0].axhline(1.0, color='red', linestyle='--', lw=2, label='Expected density')
axes[0].set_title(f'Uniformity: n={n:,}', fontweight='bold')
axes[0].set_xlabel('Value'); axes[0].set_ylabel('Density')
axes[0].legend()

# Autocorrelation check (random shouldn't correlate with itself)
lag1 = np.corrcoef(samples[:-1], samples[1:])[0,1]
axes[1].scatter(samples[:1000], np.roll(samples, -1)[:1000], alpha=0.3, s=5, color='#3498db')
axes[1].set_title(f'Lag-1 Autocorrelation: r={lag1:.4f}
(should be ≈0)', fontweight='bold')
axes[1].set_xlabel('x[i]'); axes[1].set_ylabel('x[i+1]')

# Bit pattern (test for periodicity in bits)
bits = (samples * 256).astype(int) % 2  # extract one bit
run_lengths = []
current, length = bits[0], 1
for b in bits[1:]:
    if b == current: length += 1
    else: run_lengths.append(length); current, length = b, 1
axes[2].hist(run_lengths, bins=range(1, 20), color='#27ae60', edgecolor='white', density=True)
from scipy.stats import geom
k = np.arange(1, 20)
axes[2].plot(k, geom.pmf(k, 0.5), 'ro-', markersize=5, label='Expected Geometric(0.5)')
axes[2].set_title('Run Length Distribution
(test for periodicity)', fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle("Statistical Quality Tests for NumPy's PRNG", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch13_prng_quality.png', dpi=150, bbox_inches='tight')
plt.show()

from scipy.stats import kstest
ks_stat, p_value = kstest(samples, 'uniform')
print(f"Kolmogorov-Smirnov test for uniformity: KS={ks_stat:.4f}, p={p_value:.4f}")
print("High p → looks uniform ✅" if p_value > 0.05 else "Low p → NOT uniform ❌")

In [ ]:
# ── Sampling Methods ──
rng = np.random.default_rng(42)

print("🎯 Sampling Methods")
print()
population = np.arange(1, 101)

# Simple random sampling
srs = rng.choice(population, size=10, replace=False)
print(f"Simple random sample (n=10): {sorted(srs)}")

# Weighted sampling
weights = np.linspace(1, 10, 100)
weights /= weights.sum()
weighted = rng.choice(population, size=10, replace=True, p=weights)
print(f"Weighted sample (high values more likely): {sorted(weighted)}")

# Stratified: sample equal numbers from each decile
strata = np.array_split(population, 10)
stratified = np.concatenate([rng.choice(s, 1) for s in strata])
print(f"Stratified sample (1 from each decile): {sorted(stratified)}")

# Bootstrap sample
data = np.array([23, 45, 12, 67, 34, 89, 45, 23, 56, 78])
bootstrap = rng.choice(data, size=len(data), replace=True)
print(f"
Original data: {data}")
print(f"Bootstrap sample: {bootstrap}")
print(f"(Note: some values repeated, some missing — that's bootstrap!)")

## 🔬 Section 3 — Simulation: Monte Carlo π Estimation

In [ ]:
# Monte Carlo: Estimate π using random dart throws
rng = np.random.default_rng(42)

def estimate_pi(n, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1, 1, n)
    y = rng.uniform(-1, 1, n)
    inside = (x**2 + y**2) <= 1
    return 4 * inside.mean(), x, y, inside

n_points = 10_000
pi_est, x, y, inside = estimate_pi(n_points)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Dart board visualization
ax = axes[0]
ax.scatter(x[inside],  y[inside],  s=2, c='#27ae60', alpha=0.5, label='Inside')
ax.scatter(x[~inside], y[~inside], s=2, c='#e74c3c', alpha=0.5, label='Outside')
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=2)
ax.set_aspect('equal')
ax.set_title(f'Monte Carlo π: {pi_est:.4f}
(n={n_points:,} darts)', fontweight='bold')
ax.legend(fontsize=9)

# Convergence
ns = np.logspace(1, 5, 100).astype(int)
estimates = []
for ni in ns:
    x_i = rng.uniform(-1, 1, ni)
    y_i = rng.uniform(-1, 1, ni)
    estimates.append(4 * ((x_i**2+y_i**2)<=1).mean())

axes[1].semilogx(ns, estimates, alpha=0.7, color='#3498db', label='Estimate')
axes[1].axhline(np.pi, color='red', lw=2, linestyle='--', label=f'True π={np.pi:.4f}')
axes[1].set_xlabel('Number of darts'); axes[1].set_ylabel('π estimate')
axes[1].set_title('Monte Carlo Convergence: O(1/√n) error', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch13_monte_carlo_pi.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Error ∝ 1/√n. At n={n_points}: error ≈ {abs(pi_est - np.pi):.4f}")

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Implement your own Linear Congruential Generator (simplest PRNG).
`x[n+1] = (a*x[n] + c) % m`

**Challenge 2:** Why is `random.seed(42)` at the top of an ML training script important?
Write code demonstrating model reproducibility with/without seeds.

**Challenge 3:** Implement reservoir sampling to sample k items from a stream of unknown length.

<details><summary>💡 Solutions</summary>

**Challenge 1:** See code below.

**Challenge 2:** Without seed → different weight initialization → different results every run. Seed fixes the initial weights, data shuffling, and dropout masks.

**Challenge 3:** See reservoir sampling implementation below.
</details>

In [ ]:
# Challenge 1: Linear Congruential Generator
class LCG:
    def __init__(self, seed, a=1664525, c=1013904223, m=2**32):
        self.state = seed; self.a = a; self.c = c; self.m = m
    def next(self):
        self.state = (self.a * self.state + self.c) % self.m
        return self.state / self.m
    def uniform(self, n):
        return [self.next() for _ in range(n)]

lcg = LCG(42)
samples = lcg.uniform(10000)
print(f"LCG mean: {np.mean(samples):.4f} (expected 0.5)")
print(f"LCG std:  {np.std(samples):.4f} (expected ~0.289)")

# Challenge 3: Reservoir Sampling
def reservoir_sample(stream, k, seed=42):
    rng = np.random.default_rng(seed)
    reservoir = list(stream[:k])
    for i, item in enumerate(stream[k:], start=k):
        j = rng.integers(0, i+1)
        if j < k:
            reservoir[j] = item
    return reservoir

stream = list(range(1, 10001))  # stream of 10,000 items
sample = reservoir_sample(stream, 100)
print(f"
Reservoir sample (k=100 from 10,000):")
print(f"  Mean: {np.mean(sample):.1f} (expected ~5000)")
print(f"  Min/Max: {min(sample)}/{max(sample)}")

## 🎯 Recap
1. PRNGs are deterministic — `seed` controls reproducibility.
2. Use `np.random.default_rng(seed)` for modern NumPy code.
3. Test PRNG quality with KS test, autocorrelation, and uniformity checks.

**Next:** [Chapter 14 — Monte Carlo Methods]